In [3]:
########################################################## MERGE + JOIN ################################################

In [4]:
# This is where the three datasets start talking to each other. 
# In AIOps this is critical — a spike in cpu_pct only becomes actionable 
# when you can join it to an open P1 ticket on the same server.

In [5]:
# LOAD DATASETS
import pandas as pd
import numpy as np

# ── Load all three datasets ────────────────────────────────────────────
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

# ── Confirm shapes ─────────────────────────────────────────────────────
print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [6]:
# merge() Basics + Join Types
# In pandas, merge() is the primary way to combine two DataFrames — equivalent to SQL JOIN.
# syntax - pd.merge(left_df, right_df, on="key_column", how="join_type")

In [7]:
# The 4 join types
# left_df          right_df
# A  B             A  C
# 1  x             1  p
# 2  y             3  q
# how=             Keeps                           Result
# inner      Only matching rows on both sides     safest, no nulls
# left       All left rows + matches from right   nulls where no match
# right      All right rows + matches from left   nulls where no match
# outer      All rows from both sides             most nulls

In [8]:
# INNER JOIN - (SERVER_METRICS + INCIDENTS)

In [9]:
# which servers have both metric readings and tickets ?

merged_inner = pd.merge(df_metrics, df_tickets, 
                        on="server_id", 
                        how="inner"
                       )
print(f"df_metrics rows   : {len(df_metrics)}")
print(f"df_tickets rows   : {len(df_tickets)}")
print(f"inner merge rows  : {len(merged_inner)}")
print(merged_inner[["server_id", "cpu_pct", "priority", "category"]].head() )

df_metrics rows   : 525
df_tickets rows   : 213
inner merge rows  : 22365
  server_id  cpu_pct  priority     category
0    srv-01     49.6         3     CPU High
1    srv-01     49.6         2    Disk Full
2    srv-01     49.6         2  Memory Leak
3    srv-01     49.6         2  Network Lag
4    srv-01     49.6         2  Network Lag


In [10]:
# LEFT JOIN (KEEP ALL METRICS, ATTACH TICKET INFO)

In [11]:
# Keep every metric row - attach ticket data where available
merged_left = pd.merge(df_metrics, df_tickets, on="server_id", how="left")

In [12]:
print(f"left merge rows : {len(merged_left)}")
# Check nulls on right side - rows with matching ticket
print("nulls after left join:")
print(merged_left[["priority", "category"]].isnull().sum())

left merge rows : 22365
nulls after left join:
priority    0
category    0
dtype: int64


In [13]:
# Key rules
          # Rule                                           # Why
# Always check row count after merge          Unexpected explosion = duplicate keys on one side
# Always check nulls after left/right join    Confirms match quality
# Prefer left over inner in AIOps             You never want to silently lose metric rows

In [14]:
print(df_metrics.dtypes)   
print("\n")
print(df_tickets.dtypes)

timestamp      datetime64[ns]
server_id            category
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object


ticket_id              object
server_id              object
category               object
priority                int64
created_at     datetime64[ns]
resolved_at    datetime64[ns]
resolved                 bool
dtype: object


In [15]:
# Simulate a server with metrics but no tickets
df_metrics_extra = pd.concat([
    df_metrics,
    pd.DataFrame({
        "server_id": ["srv-06"],
        "cpu_pct": [88.0],
        "memory_pct": [72.0],
        "response_ms": [500.0],
        "disk_pct": [60.0],
        "status": ["critical"],
        "timestamp": pd.Timestamp("2024-01-01")
    })
])

merged_test = pd.merge(df_metrics_extra, df_tickets, on="server_id", how="left")

print("Nulls after left join:")
print(merged_test[["priority", "category"]].isnull().sum())

# You'll see priority and category show 1 null — that's srv-06 with no matching ticket.

Nulls after left join:
priority    1
category    1
dtype: int64


In [16]:
# JOINING ON DIFFERENT KEY NAMES
# Sometimes the same concept has different column names across DataFrames. 
# merge() handles this with left_on and right_on.

In [17]:
# Using different key name host instead of server_id in df_logs

# Simulate different key names
df_logs_renamed = df_logs.rename(columns={"server_id" : "host"})

merged = pd.merge( df_metrics, df_logs_renamed, 
                   left_on = "server_id", 
                   right_on = "host",
                   how = "inner"
                 )
print(merged[["server_id", "host", "cpu_pct", "log_level"]].head())
print(f"Merged shape : {merged.shape}")

  server_id    host  cpu_pct log_level
0    srv-01  srv-01     49.6      INFO
1    srv-01  srv-01     49.6      INFO
2    srv-01  srv-01     49.6      INFO
3    srv-01  srv-01     49.6   WARNING
4    srv-01  srv-01     49.6      INFO
Merged shape : (33075, 12)


In [18]:
# Notice — both columns are kept
# After merge, you'll see both server_id and host in the result — pandas keeps both since they had different names. Drop the redundant one:

merged = merged.drop(columns=["host"])

In [19]:
# Observation 1 — Different key names merge: 33,075 rows
# Same explosion as before — joining on server_id alone is many-to-many. But you can see server_id and host both appear and match correctly. 
# Drop host after merge:

In [20]:
# Merging on Multiple Keys
# Join df_metrics and df_logs on BOTH server_id AND timestamp
# This is an exact match — same server, same timestamp
merged_exact = pd.merge(
    df_metrics,
    df_logs,
    on=["server_id", "timestamp"],
    how="inner"
)

print(f"Exact match rows: {len(merged_exact)}")
print(merged_exact[["server_id", "timestamp", "cpu_pct", "log_level"]].head())

Exact match rows: 1
  server_id           timestamp  cpu_pct log_level
0    srv-02 2026-01-02 19:00:00     82.1   WARNING


In [21]:
# Why multi-key join matters in AIOps
# Joining on server_id alone explodes rows. 
# Joining on server_id + timestamp gives you exact point-in-time correlation — 
# the CPU reading at the exact moment a log entry was written.

In [22]:
# Observation 2 — Multi-key merge: only 1 row ✅
# This is the important one. Joining on server_id + timestamp gave you exactly 1 match across the entire dataset — 
# meaning only one log entry and one metric reading share the exact same server AND exact same timestamp.

In [23]:
# So exact timestamp join is too strict in practice. The solution is time window joining — match metric readings to log entries within ±5 minutes. 

In [24]:
# MERGED INNER
merged_inner = pd.merge(
    df_metrics,
    df_tickets,
    on="server_id",
    how="inner"
)

print(f"df_metrics rows  : {len(df_metrics)}")
print(f"df_tickets rows  : {len(df_tickets)}")
print(f"inner merge rows : {len(merged_inner)}")
print(merged_inner[["server_id", "cpu_pct", "priority", "category"]].head())

df_metrics rows  : 525
df_tickets rows  : 213
inner merge rows : 22365
  server_id  cpu_pct  priority     category
0    srv-01     49.6         3     CPU High
1    srv-01     49.6         2    Disk Full
2    srv-01     49.6         2  Memory Leak
3    srv-01     49.6         2  Network Lag
4    srv-01     49.6         2  Network Lag


In [25]:
# MERGED LEFT
merged_left = pd.merge(
    df_metrics,
    df_tickets,
    on="server_id",
    how="left"
)

print(f"left merge rows  : {len(merged_left)}")
print("Nulls after left join:")
print(merged_left[["priority", "category"]].isnull().sum())

left merge rows  : 22365
Nulls after left join:
priority    0
category    0
dtype: int64


In [26]:
# DETECTING JOIN QUALITY AND THE CORRECT AIOPS MERGE PATTERN ?

In [27]:
# CHECK 1 -  ROW COUNT SANITY
print(f"left rows : {len(df_metrics)}")
print(f"right rows : {len(df_tickets)}")
print(f"Merged rows : {len(merged_inner)}")
# Rule of thumb:
# merged > left × 10 → likely a many-to-many explosion
# merged < left      → rows are being lost (check join keys)

left rows : 525
right rows : 213
Merged rows : 22365


In [28]:
# CHECK 2 - NULL AUDIT AFTER JOIN
# Always check nulls on columns coming from the right DataFrame
print("Nulls after merge:")
print(merged_left[["priority", "category", "resolved"]].isnull().sum())

# Nulls here mean left rows had no match on the right
# In AIOps — servers with metrics but no tickets

Nulls after merge:
priority    0
category    0
resolved    0
dtype: int64


In [29]:
# CHECK 3 - DUPLICATE KEY CHECK BEFORE MERGING
# Check if join key is unique on right side
print("Unique server_ids in df_tickets:")
print(df_tickets["server_id"].nunique())
print(df_tickets["server_id"].value_counts())

Unique server_ids in df_tickets:
5
server_id
srv-04    45
srv-01    45
srv-05    44
srv-02    40
srv-03    39
Name: count, dtype: int64


In [30]:
# CORRECT AIOPS MERGE PATTERN
# The fix for the explosion problem - aggregate first and then join:
# Step 1 — Summarize metrics per server (one row per server)
metrics_summary = df_metrics.groupby(
    "server_id", observed=True
).agg(
    avg_cpu      = ("cpu_pct",     "mean"),
    max_cpu      = ("cpu_pct",     "max"),
    avg_memory   = ("memory_pct",  "mean"),
    avg_response = ("response_ms", "mean")
).round(2).reset_index()

print(f"metrics_summary shape: {metrics_summary.shape}")
print(metrics_summary)

metrics_summary shape: (5, 5)
  server_id  avg_cpu  max_cpu  avg_memory  avg_response
0    srv-01    59.93     98.4       62.51        643.57
1    srv-02    59.93     98.2       64.46        633.02
2    srv-03    58.57     98.5       65.09        622.77
3    srv-04    59.41     98.6       62.21        632.40
4    srv-05    62.33     96.7       62.10        590.14


In [31]:
# Step 2 — Summarize tickets per server (one row per server)
tickets_summary = df_tickets.groupby("server_id").agg(
    total_tickets    = ("ticket_id", "count"),
    open_tickets     = ("resolved",  lambda x: (x == False).sum()),
    min_priority     = ("priority",  "min"),
    resolution_rate  = ("resolved",  lambda x: round(x.mean() * 100, 1))
).reset_index()

print(f"tickets_summary shape: {tickets_summary.shape}")
print(tickets_summary)

tickets_summary shape: (5, 5)
  server_id  total_tickets  open_tickets  min_priority  resolution_rate
0    srv-01             45            17             1             62.2
1    srv-02             40            15             1             62.5
2    srv-03             39            17             1             56.4
3    srv-04             45            18             1             60.0
4    srv-05             44            21             1             52.3


In [32]:
# Step 3 — Now merge cleanly — one row per server, no explosion
server_health = pd.merge(
    metrics_summary,
    tickets_summary,
    on="server_id",
    how="left"
)

print(f"server_health shape: {server_health.shape}")
print(server_health)

server_health shape: (5, 9)
  server_id  avg_cpu  max_cpu  avg_memory  avg_response  total_tickets  \
0    srv-01    59.93     98.4       62.51        643.57             45   
1    srv-02    59.93     98.2       64.46        633.02             40   
2    srv-03    58.57     98.5       65.09        622.77             39   
3    srv-04    59.41     98.6       62.21        632.40             45   
4    srv-05    62.33     96.7       62.10        590.14             44   

   open_tickets  min_priority  resolution_rate  
0            17             1             62.2  
1            15             1             62.5  
2            17             1             56.4  
3            18             1             60.0  
4            21             1             52.3  


In [33]:
# OBSERVATION : 
# metrics_summary → 5 rows (one per server)
# tickets_summary → 5 rows (one per server)
# server_health → 5 rows  — no explosion

# This is the production AIOps pattern — aggregate to server level first, then join. The result is a clean per-server health dashboard.

In [34]:
# health dashboard
  # Server                 # Concern
  # srv-05               Highest avg CPU (62.33) + most open tickets (21) + lowest resolution rate (52.3%) — highest risk
  # srv-03               Lowest resolution rate (56.4%) — tickets hard to close
  # srv-01               Highest avg response (643ms) — slowest server
  # srv-02               Most balanced overall
  # srv-04               Most open tickets after srv-05 (18)

# The pattern to remember
# merge on raw DataFrames → explosion
# Best way - groupby → agg → reset_index → merge → clean
# reset_index() is critical here — it converts the groupby index back to a regular column so merge() can find server_id as a normal column.

In [35]:
# CONCAT() VS MERGE()

In [36]:
# concat — stacks DataFrames vertically (more rows)
# pd.concat([df1, df2])          # same columns, add rows

# merge — combines DataFrames horizontally (more columns)
# pd.merge(df1, df2, on="key")   # join on common key, add columns

In [37]:
# Real example — stack two months of metrics
df_metrics_feb = df_metrics.copy()  # simulate second month
combined = pd.concat([df_metrics, df_metrics_feb], ignore_index=True)
print(f"Combined shape: {combined.shape}")  # 1050 rows

Combined shape: (1050, 7)


In [38]:
# LAB TASK
# Task 1 — server_metrics + incidents
# Aggregate both DataFrames to server level, then merge. Add a risk_score column:
# risk_score = (avg_cpu * 0.4) + (open_tickets * 0.4) + (avg_response / 100 * 0.2)
# Sort by risk_score descending. Which server is highest risk?

In [39]:
df_metrics.dtypes

timestamp      datetime64[ns]
server_id            category
cpu_pct               float64
memory_pct            float64
response_ms           float64
disk_pct              float64
status                 object
dtype: object

In [40]:
df_tickets.dtypes

ticket_id              object
server_id              object
category               object
priority                int64
created_at     datetime64[ns]
resolved_at    datetime64[ns]
resolved                 bool
dtype: object

In [41]:
metrics_summ = df_metrics.groupby("server_id", observed=True).agg(
    avg_cpu = ("cpu_pct", "mean"),
    avg_response = ("response_ms", "mean")
).reset_index()
print(metrics_summ)

  server_id    avg_cpu  avg_response
0    srv-01  59.926667    643.574286
1    srv-02  59.931429    633.023810
2    srv-03  58.565714    622.768571
3    srv-04  59.409524    632.403810
4    srv-05  62.332381    590.140952


In [42]:
tickets_summ = df_tickets.groupby("server_id").agg(
    total_tickets = ("ticket_id", "count"),
    open_tickets = ("resolved", lambda x: (x== False).sum())
).reset_index()
print(tickets_summ)

  server_id  total_tickets  open_tickets
0    srv-01             45            17
1    srv-02             40            15
2    srv-03             39            17
3    srv-04             45            18
4    srv-05             44            21


In [43]:
server_merge = pd.merge(metrics_summ, tickets_summ, on="server_id", how="left")
print(server_merge)

  server_id    avg_cpu  avg_response  total_tickets  open_tickets
0    srv-01  59.926667    643.574286             45            17
1    srv-02  59.931429    633.023810             40            15
2    srv-03  58.565714    622.768571             39            17
3    srv-04  59.409524    632.403810             45            18
4    srv-05  62.332381    590.140952             44            21


In [46]:
server_merge["risk_score"] = (server_merge["avg_cpu"] * 0.4 + 
                              server_merge["open_tickets"] * 0.4 + 
                              server_merge["avg_response"] / 100 * 0.2
                             ).round(2)

In [47]:
print(server_merge.sort_values("risk_score", ascending=False))

  server_id    avg_cpu  avg_response  total_tickets  open_tickets  risk_score
4    srv-05  62.332381    590.140952             44            21       34.51
3    srv-04  59.409524    632.403810             45            18       32.23
0    srv-01  59.926667    643.574286             45            17       32.06
2    srv-03  58.565714    622.768571             39            17       31.47
1    srv-02  59.931429    633.023810             40            15       31.24


In [48]:
# # OBSERVATION: server-5 is highest risk score 34.51 driven by Highest cpu 62.33 and most open tickets (21)
# One thing to note
# avg_response contribution to risk score is very small because of the /100 * 0.2 scaling. 
# srv-01 has the highest response time (643ms) but it barely moves its score. 
# In a real AIOps risk model you'd normalize response_ms to 0-1 range first before weighting. 
# That's a related to Topic (Feature Engineering) pattern.